# Notebook 3a — Type Ia Supernovae

Type Ia supernovae are standard candles — objects with a known intrinsic luminosity. By comparing their observed brightness to their theoretical luminosity distance, we can constrain cosmological parameters.

The observable is the **apparent magnitude** $m_b$, related to the luminosity distance by:

$$m_b = 5\log_{10}\left(d_L(z)\right) + \mathcal{M}$$

where $\mathcal{M} = M + 25 + 5\log_{10}(D_H)$ absorbs the absolute magnitude $M$ and the Hubble distance $D_H = c/H_0$.

We use the **Pantheon** sample — 1048 supernovae from $z \sim 0.01$ to $z \sim 1.3$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from plots import *
from chi2 import chi2_sne, chi2_sne_full, load_sne
from tqdm.notebook import tqdm


---
## 1. Load the data `[S1]`

In [ ]:
z_sn, mb, dmb = load_sne()
print(f"Loaded {len(z_sn)} supernovae")
print(f"Redshift range: {z_sn.min():g} — {z_sn.max():g}")

---
## 2. Full $\chi^2$ — scanning $(\Omega_m, \mathcal{M})$ `[S2]`

The full $\chi^2$ has two free parameters:

$$\chi^2(\Omega_m, \mathcal{M}) = \sum_i \frac{\left(m_b^i - 5\log_{10}(d_L(z_i)) - \mathcal{M}\right)^2}{\sigma_i^2}$$

We scan over both to see how they are constrained — and how degenerate they are.

In [ ]:
from tqdm.notebook import tqdm

Om_arr    = np.linspace(0.2, 0.4, 60)
Mcurl_arr = np.linspace(23.7, 23.9, 60)
chi2_grid = np.zeros((len(Om_arr), len(Mcurl_arr)))

for i, Om in enumerate(Om_arr):
    rho_de, _ = de_model('LCDM', Om)
    # compute distances once per Om — not inside the Mcurl loop
    z_th, _, d_L_th = solve_distances(Om, rho_de, z_max=z_sn.max()*1.01,
                                       n_points=1000)
    d_L_data = np.interp(z_sn, z_th, d_L_th)
    mu_base  = 5 * np.log10(d_L_data)
    # inner loop is now pure arithmetic
    for j, Mcurl in enumerate(Mcurl_arr):
        delta          = mb - mu_base - Mcurl
        chi2_grid[i,j] = np.sum((delta / dmb)**2)

chi2_min = chi2_grid.min()
idx = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)
Om_best_2d    = Om_arr[idx[0]]
Mcurl_best_2d = Mcurl_arr[idx[1]]

print(f"Best fit Om    = {Om_best_2d:g}")
print(f"Best fit Mcurl = {Mcurl_best_2d:g}")
print(f"chi2_min       = {chi2_min:g}")

fig, ax = plt.subplots(figsize=(7, 6))
ct = ax.contour(Om_arr, Mcurl_arr, (chi2_grid - chi2_min).T,
                levels=[2.30, 6.17], colors=['#2a78d6', '#e34948'])
ax.clabel(ct, fmt={2.30: r'$1\sigma$', 6.17: r'$2\sigma$'}, fontsize=10)
ax.plot(Om_best_2d, Mcurl_best_2d, 'k*', ms=10, label='best fit')
ax.set_xlabel(r'$\Omega_m$')
ax.set_ylabel(r'$\mathcal{M}$')
ax.set_title(r'SNe Ia — full $\chi^2$')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 3. Marginalised $\chi^2$ `[S3]`

Since $\mathcal{M}$ is a nuisance parameter, we can marginalise over it analytically. The marginalised $\chi^2$ is:

$$\chi^2 = A - \frac{B^2}{C}$$

where $\Delta_i = m_b^i - 5\log_{10}(d_L(z_i))$ and:

$$A = \sum_i \frac{\Delta_i^2}{\sigma_i^2} \qquad B = \sum_i \frac{\Delta_i}{\sigma_i^2} \qquad C = \sum_i \frac{1}{\sigma_i^2}$$

This reduces the fit to $\Omega_m$ only.

In [ ]:
Om_arr   = np.linspace(0.1, 0.6, 100)
chi2_arr = np.zeros(len(Om_arr))
Mcurl_arr = np.zeros(len(Om_arr))

for i, Om in enumerate(Om_arr):
    rho_de, _ = de_model('LCDM', Om)
    chi2_arr[i], Mcurl_arr[i] = chi2_sne(Om, rho_de, z_sn, mb, dmb)

idx_min    = np.argmin(chi2_arr)
Om_best    = Om_arr[idx_min]
chi2_min   = chi2_arr[idx_min]
Mcurl_best = Mcurl_arr[idx_min]

print(f"Best fit Om    = {Om_best:g}")
print(f"Best fit Mcurl = {Mcurl_best:g}")
print(f"chi2_min       = {chi2_min:g}")
print(f"chi2/dof       = {chi2_min/(len(z_sn)-1):g}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(Om_arr, chi2_arr - chi2_min, color='#2a78d6')
ax.axvline(Om_best, color='#e34948', ls='--', lw=1.5,
           label=f'$\\Omega_m = {Om_best:g}$')
ax.axhline(1, color='black', ls=':', lw=1, alpha=0.5, label=r'$\Delta\chi^2 = 1$')
ax.set_xlabel(r'$\Omega_m$')
ax.set_ylabel(r'$\Delta\chi^2$')
ax.set_title(r'SNe Ia — marginalised $\chi^2$')
ax.set_ylim(0, 20)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 4. Data vs best-fit theory `[S4]`

In [ ]:
rho_de, _ = de_model('LCDM', Om_best)

z_max_plot = z_sn.max() * 1.1
z_th, _, d_L_th = solve_distances(Om_best, rho_de, z_max=z_max_plot, n_points=500)
mu_th = 5 * np.log10(d_L_th) + Mcurl_best

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(z_sn, mb, yerr=dmb, fmt='.', color='#e34948',
            alpha=0.6, ms=3, lw=0.5, label='Pantheon data')
ax.plot(z_th, mu_th, color='#2a78d6', lw=2,
        label=f'ΛCDM ($\\Omega_m={Om_best:g}$)')
ax.set_xlabel(r'redshift $z$')
ax.set_ylabel(r'apparent magnitude $m_b$')
ax.set_title('Pantheon SNe Ia — data vs best-fit ΛCDM')
ax.set_ylim(mb.min() - 0.5, mb.max() + 0.5)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 5. Extracting $H_0$ `[S5]`

The best-fit $\mathcal{M}$ encodes $H_0$ via:

$$\mathcal{M} = M + 25 + 5\log_{10}\left(\frac{c}{H_0}\right) \implies H_0 = \frac{c}{10^{(\mathcal{M} - M - 25)/5}}$$

Assuming $M = -19.3$ (SH0ES Cepheid calibration):

In [ ]:
M_calib = -19.3
H0 = 3e5 / 10**((Mcurl_best - M_calib - 25) / 5)

print(f"Best fit Om = {Om_best:g}")
print(f"Mcurl       = {Mcurl_best:g}")
print(f"Assuming M  = {M_calib:g}")
print(f"=> H0       = {H0:.1f} km/s/Mpc")